In [ ]:
# RL-Powered Content Format Converter Demo

This notebook demonstrates the complete conversion pipeline with examples of all supported conversions.

## Prerequisites

1. Start the FastAPI backend:
   ```bash
   cd backend
   uvicorn main:app --host 0.0.0.0 --port 8000 --reload
   ```

2. Install required Python packages:
   ```bash
   pip install requests matplotlib librosa
   ```

## Overview

The RL-powered converter supports:
- **Video → Audio**: Extract audio tracks from video files
- **Audio → Text**: Transcribe speech using Whisper STT
- **Text → Audio**: Generate speech using TTS
- **Video → Text**: Complete pipeline (video → audio → text)
- **Text → Video**: Generate video summaries (stub)

Each conversion includes:
- RL agent decision making
- Quality assessment and scoring
- Metadata enrichment
- Schema-aligned output for team integration



SyntaxError: invalid character '→' (U+2192) (1552067442.py, line 21)

In [ ]:
import requests
import json
import matplotlib.pyplot as plt
import os
from pathlib import Path

# Configuration
URL = 'http://localhost:8000/convert-content'
API_BASE = 'http://localhost:8000'

print("🚀 RL-Powered Content Converter Demo")
print("=" * 50)


In [ ]:
# Test API Health
def test_health():
    """Test if the API is running."""
    try:
        response = requests.get(f"{API_BASE}/health")
        if response.status_code == 200:
            print("✅ API is healthy and running")
            return True
        else:
            print(f"❌ API health check failed: {response.status_code}")
            return False
    except Exception as e:
        print(f"❌ Cannot connect to API: {e}")
        return False

test_health()


In [ ]:
# Text to Audio Conversion
def text_to_audio(text):
    """Convert text to audio using TTS."""
    print(f"\n🎤 Converting text to audio: '{text[:50]}...'")
    
    data = {
        'input_type': 'text',
        'output_type': 'audio',
        'input_text': text
    }
    
    response = requests.post(URL, data=data)
    
    if response.status_code == 200:
        # Check if it's a file download
        if response.headers.get('content-type') == 'audio/mpeg':
            print("✅ Audio generated successfully")
            print(f"📁 File size: {len(response.content)} bytes")
            
            # Save the audio file
            with open('demo_output.wav', 'wb') as f:
                f.write(response.content)
            print("💾 Audio saved as 'demo_output.wav'")
        else:
            # JSON response
            result = response.json()
            print("✅ Conversion completed")
            print(f"📊 Metadata: {json.dumps(result.get('metadata', {}), indent=2)}")
            print(f"🤖 RL Action: {result.get('rl', {}).get('action', 'unknown')}")
    else:
        print(f"❌ Conversion failed: {response.status_code}")
        print(response.text)

# Example 1: Text to Audio
text_to_audio("Hello! This is a demonstration of the RL-powered content converter. The system uses reinforcement learning to optimize conversion quality.")


In [ ]:
# Audio to Text Conversion (if sample audio exists)
def audio_to_text(audio_path):
    """Convert audio to text using Whisper STT."""
    if not os.path.exists(audio_path):
        print(f"❌ Audio file not found: {audio_path}")
        return None
        
    print(f"\n🎧 Converting audio to text: {audio_path}")
    
    with open(audio_path, 'rb') as f:
        files = {'input_file': (os.path.basename(audio_path), f, 'audio/wav')}
        data = {
            'input_type': 'audio',
            'output_type': 'text'
        }
        
        response = requests.post(URL, data=data, files=files)
    
    if response.status_code == 200:
        result = response.json()
        print("✅ Transcription completed")
        print(f"📝 Transcript: {result.get('transcript', 'No transcript')}")
        print(f"🌍 Language: {result.get('metadata', {}).get('language', 'unknown')}")
        print(f"⏱️ Duration: {result.get('metadata', {}).get('duration', 0)} seconds")
        print(f"🎯 Clarity Score: {result.get('metadata', {}).get('clarity_score', 0)}")
        return result
    else:
        print(f"❌ Transcription failed: {response.status_code}")
        print(response.text)
        return None

# Example 2: Audio to Text (if sample exists)
sample_audio = "../backend/temp/sample.wav"
audio_to_text(sample_audio)


In [ ]:
# Video to Text Conversion (Complete Pipeline)
def video_to_text(video_path):
    """Convert video to text using complete pipeline."""
    if not os.path.exists(video_path):
        print(f"❌ Video file not found: {video_path}")
        return None
        
    print(f"\n🎬 Converting video to text: {video_path}")
    
    with open(video_path, 'rb') as f:
        files = {'input_file': (os.path.basename(video_path), f, 'video/mp4')}
        data = {
            'input_type': 'video',
            'output_type': 'text'
        }
        
        response = requests.post(URL, data=data, files=files)
    
    if response.status_code == 200:
        result = response.json()
        print("✅ Video transcription completed")
        print(f"📝 Transcript: {result.get('transcript', 'No transcript')}")
        print(f"🌍 Language: {result.get('metadata', {}).get('language', 'unknown')}")
        print(f"⏱️ Duration: {result.get('metadata', {}).get('duration', 0)} seconds")
        print(f"🎯 Clarity Score: {result.get('metadata', {}).get('clarity_score', 0)}")
        print(f"🤖 RL Action: {result.get('rl', {}).get('action', 'unknown')}")
        return result
    else:
        print(f"❌ Video transcription failed: {response.status_code}")
        print(response.text)
        return None

# Example 3: Video to Text (if sample exists)
sample_video = "../backend/temp/sam.mp4"
video_to_text(sample_video)


In [ ]:
# RL Agent Performance Analysis
def analyze_rl_performance():
    """Analyze RL agent performance and learning."""
    print("\n🤖 RL Agent Performance Analysis")
    print("=" * 40)
    
    # Make multiple conversions to see RL learning
    test_texts = [
        "Short test",
        "This is a medium length test with more content",
        "This is a very long test with extensive content that should trigger different RL behaviors and parameter adjustments"
    ]
    
    results = []
    for i, text in enumerate(test_texts):
        print(f"\n🔄 Test {i+1}: {len(text)} characters")
        
        data = {
            'input_type': 'text',
            'output_type': 'audio',
            'input_text': text
        }
        
        response = requests.post(URL, data=data)
        if response.status_code == 200:
            result = response.json()
            rl_data = result.get('rl', {})
            metadata = result.get('metadata', {})
            
            results.append({
                'text_length': len(text),
                'clarity_score': metadata.get('clarity_score', 0),
                'reward': result.get('reward', 0),
                'exploration_epsilon': rl_data.get('exploration_epsilon', 0),
                'learning_rate': rl_data.get('learning_rate', 0),
                'action': rl_data.get('action', 'unknown')
            })
            
            print(f"  Action: {rl_data.get('action', 'unknown')}")
            print(f"  Clarity: {metadata.get('clarity_score', 0):.2f}")
            print(f"  Reward: {result.get('reward', 0):.2f}")
            print(f"  Exploration: {rl_data.get('exploration_epsilon', 0):.3f}")
    
    return results

# Run RL performance analysis
rl_results = analyze_rl_performance()


In [ ]:
# Schema Integration Demo
def demonstrate_schema_integration():
    """Demonstrate schema alignment for team integration."""
    print("\n🔗 Schema Integration Demo")
    print("=" * 30)
    
    # Example conversion
    data = {
        'input_type': 'text',
        'output_type': 'audio',
        'input_text': 'This demonstrates the schema alignment for Ashmit\'s backend integration.'
    }
    
    response = requests.post(URL, data=data)
    if response.status_code == 200:
        result = response.json()
        
        print("✅ Schema-aligned output structure:")
        print(f"📄 Transcript: {result.get('transcript')}")
        print(f"🎵 Generated Audio: {result.get('generated_audio')}")
        print(f"🎬 Generated Video: {result.get('generated_video')}")
        
        print("\n📊 Feedback structure:")
        feedback = result.get('feedback', {})
        print(f"  Clarity Score: {feedback.get('clarity_score')}")
        print(f"  Reward: {feedback.get('reward')}")
        print(f"  User Rating: {feedback.get('user_rating')}")
        print(f"  Suggestions: {feedback.get('improvement_suggestions')}")
        
        print("\n🤖 RL metadata:")
        rl = result.get('rl', {})
        print(f"  State: {rl.get('state')}")
        print(f"  Action: {rl.get('action')}")
        print(f"  Exploration: {rl.get('exploration_epsilon')}")
        print(f"  Learning Rate: {rl.get('learning_rate')}")
        
        return result
    else:
        print(f"❌ Schema demo failed: {response.status_code}")
        return None

# Run schema integration demo
schema_demo = demonstrate_schema_integration()


## Summary

This demo showcases the RL-powered content converter with:

✅ **Complete Conversion Pipeline**: All supported format conversions  
✅ **RL Learning**: Adaptive parameter adjustment based on performance  
✅ **Schema Alignment**: Team-ready output structure for backend integration  
✅ **Metadata Enrichment**: Comprehensive quality metrics and metadata  
✅ **Error Handling**: Robust error handling and validation  
✅ **Health Monitoring**: API health checks and monitoring  

## Next Steps

1. **Deploy to Production**: Use the Dockerfile and CI/CD pipeline
2. **Team Integration**: Connect with Ashmit's backend using the schema-aligned outputs
3. **Performance Monitoring**: Track RL agent learning and conversion quality
4. **User Feedback**: Implement feedback collection for continuous improvement

## API Usage Examples

### cURL Examples

```bash
# Text to Audio
curl -X POST "http://localhost:8000/convert-content" \
  -F "input_type=text" \
  -F "output_type=audio" \
  -F "input_text=Hello world"

# Audio to Text
curl -X POST "http://localhost:8000/convert-content" \
  -F "input_file=@sample.wav" \
  -F "input_type=audio" \
  -F "output_type=text"

# Video to Text
curl -X POST "http://localhost:8000/convert-content" \
  -F "input_file=@video.mp4" \
  -F "input_type=video" \
  -F "output_type=text"
```

### Python Client

```python
import requests

def convert_content(input_type, output_type, input_text=None, input_file=None):
    data = {
        'input_type': input_type,
        'output_type': output_type
    }
    
    if input_text:
        data['input_text'] = input_text
    
    files = None
    if input_file:
        files = {'input_file': input_file}
    
    response = requests.post('http://localhost:8000/convert-content', 
                           data=data, files=files)
    return response.json()
```
